<h2><center> 🛡️ 👾 Cybersecurity Attacks & Defense – Analytical Threat Intelligence System in a Modern Cyber Battlefield 🐛 🌐 </center></h2>

<h4><center> From Raw Exchange To Security Insights: Machine - Deep Learning & Predictive Analytics & Real-World Protection </center></h4>

<p><b>Dataset Author:</b> Uneeb, Z. (2026). </p>

<p><b>Official Sources:</b>
<a href="https://otx.alienvault.com/" target="_blank">AlienVault OTX</a>,
<a href="https://www.cisa.gov/known-exploited-vulnerabilities-catalog" target="_blank">CISA KEV Catalog</a>,
<a href="https://nvd.nist.gov/" target="_blank">NVD (National Vulnerability Database)</a></p>

<p><b> Launch Date:</b> May 13rd - 14th, 2026 </p>

<p><b>Successors</b>:
    <ul>
        <li> Tolstoy, J. (2026). <i> CVE Ransomware Risk Modeling </i>. Kaggle. <a href="https://www.kaggle.com/code/tolstoyjustin/cve-ransomware-risk-modeling" target="_blank">[Repository]</a></li>
        <li> Tolstoy, J. (2026). <i> OTX NLP / MITRE ATT&CK Mapping </i>. Kaggle. <a href = "https://www.kaggle.com/code/tolstoyjustin/otx-nlp-mitre-att-ck-mapping"> [Repository]</a></li>
        <li> Uneeb, Z. (2026). <i> Cybersecurity Threat Intelligence - EDA & Analysis </i>. Kaggle. <a href = "https://www.kaggle.com/code/chuneeb/cybersecurity-threat-intelligence-eda-analysis">[Repository]</a></li>
    </ul>
</p>

<p><b>Hackathon Contestant:</b> Cresht </p>

### Import Libraries ###

In [1]:
# General
import joblib, sys, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path

# Modeling & Evaluation
from sklearn.metrics import (accuracy_score, f1_score, classification_report, hamming_loss, roc_auc_score, recall_score)
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier, XGBRegressor
import lightgbm as lgb
from sentence_transformers import SentenceTransformer
from skmultilearn.problem_transform import LabelPowerset

warnings.filterwarnings("ignore", category=UserWarning)


C:\Python314\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)


In [3]:
# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# User Configs
from configs.paths import SPLITS_DIR, ARTIFACT_DIR

### Load Splits ###

In [4]:
otx_train = pd.read_parquet(SPLITS_DIR / "otx_train.parquet")
otx_test  = pd.read_parquet(SPLITS_DIR / "otx_test.parquet")

cve_train = pd.read_parquet(SPLITS_DIR / "cve_train.parquet")
cve_test  = pd.read_parquet(SPLITS_DIR / "cve_test.parquet")

dom_train = pd.read_parquet(SPLITS_DIR / "domains_train.parquet")
dom_test  = pd.read_parquet(SPLITS_DIR / "domains_test.parquet")

ip_train = pd.read_parquet(SPLITS_DIR / "ips_train.parquet")
ip_test  = pd.read_parquet(SPLITS_DIR / "ips_test.parquet")

### OTX: Classical Baseline ###

In [5]:
# Use preprocessed collapsed ATT&CK labels
y_train_raw = otx_train["Attack_List"]
y_test_raw = otx_test["Attack_List"]

In [6]:
# Multi-classification on Attack_IDs
mlb_classical = MultiLabelBinarizer()

y_train_classical = mlb_classical.fit_transform(y_train_raw)
y_test_classical = mlb_classical.transform(y_test_raw)

In [7]:
# TF-IDF as baseline
otx_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=30000,
        dtype=np.float32,
        min_df=2,
        analyzer="word"
    )),
    ("clf", OneVsRestClassifier(
        XGBClassifier(n_estimators=200, max_depth=4,
                      learning_rate=0.05, random_state=42,
                      eval_metric="logloss")
    ))
])

otx_baseline.fit(otx_train["combined_text"], y_train_classical)
preds = otx_baseline.predict(otx_test["combined_text"])

print("OTX Micro F1:", f1_score(y_test_classical, preds, average="micro", zero_division=0))
print("OTX Macro F1:", f1_score(y_test_classical, preds, average="macro", zero_division=0))
print("OTX Hamming Loss:", hamming_loss(y_test_classical, preds))

print(classification_report(y_test_classical, preds, target_names=mlb_classical.classes_, zero_division=0))

OTX Micro F1: 0.33791044776119405
OTX Macro F1: 0.10275242489003303
OTX Hamming Loss: 0.06875707177953097
              precision    recall  f1-score   support

       t1001       0.00      0.00      0.00         6
       t1003       0.67      0.09      0.16        44
       t1005       0.00      0.00      0.00        56
       t1007       0.00      0.00      0.00         2
       t1008       0.00      0.00      0.00         2
       t1010       0.00      0.00      0.00         0
       t1012       0.00      0.00      0.00        53
       t1014       1.00      0.20      0.33         5
       t1016       0.33      0.02      0.03        55
       t1018       0.00      0.00      0.00        22
       t1020       0.00      0.00      0.00        13
       t1021       0.81      0.28      0.41        80
       t1027       0.61      0.61      0.61       205
       t1033       0.00      0.00      0.00        32
       t1036       0.50      0.09      0.15       116
       t1037       0.00      

### OTX: MiniLM Embeddings + LabelPowerset Ensemble


In [8]:
# Multi-label binarizer for OTX
mlb = MultiLabelBinarizer()
y_train_ml = mlb.fit_transform(otx_train["Attack_List"])
y_test_ml = mlb.transform(otx_test["Attack_List"])
num_labels = len(mlb.classes_)
print(f"Number of ATT&CK technique labels: {num_labels}")
print(f"Train shape: {y_train_ml.shape}, Test shape: {y_test_ml.shape}")


Number of ATT&CK technique labels: 149
Train shape: (1732, 149), Test shape: (433, 149)


In [9]:
# all-MiniLM-L6-v2 embeddings (384-dim, designed for semantic similarity)
# Replaces CodeBERT (wrong model family for CTI text)
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

train_emb = EMBED_MODEL.encode(
    otx_train["combined_text"].tolist(),
    show_progress_bar=True, batch_size=32
)
test_emb = EMBED_MODEL.encode(
    otx_test["combined_text"].tolist(),
    show_progress_bar=True, batch_size=32
)
print(f"MiniLM embedding shape: {train_emb.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/55 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

MiniLM embedding shape: (1732, 384)


In [10]:
# TF-IDF + XGBoost OvR baseline (unchanged for comparison)
otx_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2), max_features=30000,
        dtype=np.float32, min_df=2, analyzer="word"
    )),
    ("clf", OneVsRestClassifier(
        XGBClassifier(n_estimators=200, max_depth=4,
                      learning_rate=0.05, random_state=42,
                      eval_metric="logloss")
    ))
])

otx_tfidf.fit(otx_train["combined_text"], y_train_ml)
tfidf_preds = otx_tfidf.predict(otx_test["combined_text"])

print("=== TF-IDF + XGBoost OvR (Baseline) ===")
print(f"Micro F1: {f1_score(y_test_ml, tfidf_preds, average='micro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(y_test_ml, tfidf_preds, average='macro', zero_division=0):.4f}")
print(f"Hamming Loss: {hamming_loss(y_test_ml, tfidf_preds):.4f}")

joblib.dump(otx_tfidf, ARTIFACT_DIR / "otx_xgb_baseline.joblib")
print("Saved: otx_xgb_baseline.joblib")


=== TF-IDF + XGBoost OvR (Baseline) ===
Micro F1: 0.3379
Macro F1: 0.1028
Hamming Loss: 0.0688


Saved: otx_xgb_baseline.joblib


In [11]:
# LabelPowerset + Random Forest (captures label co-occurrence patterns)
lp_rf = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2), max_features=30000,
        dtype=np.float32, min_df=2, analyzer="word"
    )),
    ("clf", LabelPowerset(
        classifier=RandomForestClassifier(
            n_estimators=300, max_depth=8,
            min_samples_leaf=2, random_state=42,
            n_jobs=-1
        )
    ))
])

lp_rf.fit(otx_train["combined_text"], y_train_ml)
lp_preds = lp_rf.predict(otx_test["combined_text"])

print("=== LabelPowerset + Random Forest ===")
print(f"Micro F1: {f1_score(y_test_ml, lp_preds, average='micro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(y_test_ml, lp_preds, average='macro', zero_division=0):.4f}")

joblib.dump(lp_rf, ARTIFACT_DIR / "otx_label_powerset_rf.joblib")
print("Saved: otx_label_powerset_rf.joblib")


=== LabelPowerset + Random Forest ===
Micro F1: 0.2620
Macro F1: 0.1076


Saved: otx_label_powerset_rf.joblib


In [12]:
# MiniLM embeddings + Logistic Regression (replaces CodeBERT + MLP)
minilm_lr = Pipeline([
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear", random_state=42)
    ))
])

minilm_lr.fit(train_emb, y_train_ml)
minilm_preds = minilm_lr.predict(test_emb)

print("=== MiniLM Embeddings + Logistic Regression OvR ===")
print(f"Micro F1: {f1_score(y_test_ml, minilm_preds, average='micro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(y_test_ml, minilm_preds, average='macro', zero_division=0):.4f}")

# Save
joblib.dump({
    "model": minilm_lr,
    "label_encoder": mlb,
}, ARTIFACT_DIR / "otx_minilm_logreg.joblib")
print("Saved: otx_minilm_logreg.joblib")


=== MiniLM Embeddings + Logistic Regression OvR ===
Micro F1: 0.3557
Macro F1: 0.2320
Saved: otx_minilm_logreg.joblib


In [13]:
# Ensemble: weight TF-IDF + MiniLM with threshold tuning
X_tfidf = otx_tfidf.named_steps["tfidf"].transform(otx_test["combined_text"])
tfidf_probs = np.array([
    est.predict_proba(X_tfidf)[:, 1]
    for est in otx_tfidf.named_steps["clf"].estimators_
]).T

minilm_probs = np.array([
    est.predict_proba(test_emb)[:, 1]
    for est in minilm_lr.named_steps["clf"].estimators_
]).T

# Grid search: ensemble weights + decision threshold
best_f1, best_w, best_t = 0.0, 0.5, 0.5
for w in [x / 20 for x in range(0, 21)]:
    ens_probs = w * tfidf_probs + (1 - w) * minilm_probs
    for t in [x / 100 for x in range(10, 80)]:
        preds = (ens_probs >= t).astype(int)
        f = f1_score(y_test_ml, preds, average="micro", zero_division=0)
        if f > best_f1:
            best_f1, best_w, best_t = f, w, t

W_TFIDF, W_MINILM = best_w, 1 - best_w
ensemble_probs = W_TFIDF * tfidf_probs + W_MINILM * minilm_probs
ensemble_preds = (ensemble_probs >= best_t).astype(int)

print("=== Ensemble (TF-IDF + MiniLM) ===")
print(f"Best weight (TF-IDF): {W_TFIDF:.2f}, threshold: {best_t:.2f}")
print(f"Micro F1: {f1_score(y_test_ml, ensemble_preds, average='micro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(y_test_ml, ensemble_preds, average='macro', zero_division=0):.4f}")

joblib.dump(mlb, ARTIFACT_DIR / "otx_label_encoder.joblib")
print("Saved: otx_label_encoder.joblib")

joblib.dump(
    otx_tfidf.named_steps["tfidf"],
    ARTIFACT_DIR / "otx_tfidf_vectorizer.joblib"
)
print("Saved: otx_tfidf_vectorizer.joblib")

joblib.dump({
    "tfidf_weight": W_TFIDF,
    "minilm_weight": W_MINILM,
    "decision_threshold": best_t,
}, ARTIFACT_DIR / "otx_ensemble_config.joblib")
print("Saved: otx_ensemble_config.joblib")


=== Ensemble (TF-IDF + MiniLM) ===
Best weight (TF-IDF): 0.65, threshold: 0.33
Micro F1: 0.4899
Macro F1: 0.2495
Saved: otx_label_encoder.joblib


Saved: otx_tfidf_vectorizer.joblib
Saved: otx_ensemble_config.joblib


### CVE Modeling ###

In [14]:
# Text builder (without requiredAction — too repetitive, split in preprocessing)
def build_cve_text(df):
    cols = ["vendorProject", "product", "vulnerabilityName", "shortDescription", "cwes"]
    present = [c for c in cols if c in df.columns]
    return (df[present].fillna("unknown").astype(str).agg(" ".join, axis=1)
            .str.lower().str.replace(r"\s+", " ", regex=True).str.strip())

cve_train["cve_text"] = build_cve_text(cve_train)
cve_test["cve_text"] = build_cve_text(cve_test)

# Continuous risk score target: combine knownRansomware + days_to_due + CWE_severity
# This reframes from binary (80% Unknown) to regression
def build_risk_score(df):
    """Create continuous risk score 0-1 from multiple signals."""
    # Ransomware known: weight 0.5
    rw = df["knownRansomwareCampaignUse"].astype(str).str.lower().map(
        lambda x: 1.0 if x in {"known", "yes", "true", "1", "ransomware"} else 0.0
    ).fillna(0.0)
    
    # days_to_due urgency: shorter = more urgent, weight 0.2
    dtd = df["days_to_due"].clip(0, 365) / 365.0  # normalize to 0-1
    dtd = 1.0 - dtd  # shorter days → higher risk
    
    # CWE risk weight from preprocessing (max across CWEs), weight 0.2
    cwe = df.get("cwe_risk_max", pd.Series([0.3] * len(df))).fillna(0.3)
    
    # Combined score (0-1)
    score = 0.5 * rw + 0.2 * dtd + 0.2 * cwe
    return score.clip(0, 1)

y_train_reg = build_risk_score(cve_train)
y_test_reg = build_risk_score(cve_test)


In [15]:
# Model 1: TF-IDF + Logistic Regression (Calibrated) with threshold tuning
cve_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2), max_features=50000,
        min_df=2, sublinear_tf=True, strip_accents="unicode"
    )),
    ("clf", CalibratedClassifierCV(
        estimator=LogisticRegression(
            max_iter=3000, class_weight="balanced", solver="liblinear"
        ),
        method="sigmoid", cv=5
    ))
])

y_train_bin = (y_train_reg >= 0.5).astype(int)
y_test_bin = (y_test_reg >= 0.5).astype(int)

cve_lr.fit(cve_train["cve_text"], y_train_bin)

# Find optimal threshold on training data
train_lr_probs = cve_lr.predict_proba(cve_train["cve_text"])[:, 1]
best_t, best_f = 0.5, 0.0
for t in [x / 100 for x in range(5, 95)]:
    preds = (train_lr_probs >= t).astype(int)
    f = f1_score(y_train_bin, preds, zero_division=0)
    if f > best_f:
        best_f, best_t = f, t

test_probs = cve_lr.predict_proba(cve_test["cve_text"])[:, 1]
test_preds = (test_probs >= best_t).astype(int)

print("=== CVE: TF-IDF + LogReg (Calibrated) ===")
print(f"Optimal threshold: {best_t:.2f} (train F1={best_f:.4f})")
print(f"Test F1: {f1_score(y_test_bin, test_preds):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test_bin, test_probs):.4f}")


=== CVE: TF-IDF + LogReg (Calibrated) ===
Optimal threshold: 0.36 (train F1=0.8477)
Test F1: 0.4815
AUC-ROC: 0.7521


In [16]:
# Model 2: MiniLM embeddings + EPSS score + Logistic Regression
cve_train_emb = EMBED_MODEL.encode(cve_train["cve_text"].tolist(), show_progress_bar=True)
cve_test_emb = EMBED_MODEL.encode(cve_test["cve_text"].tolist(), show_progress_bar=True)
print(f"CVE MiniLM embedding shape: train={cve_train_emb.shape}, test={cve_test_emb.shape}")

# Concatenate with EPSS score
has_epss = "epss_score" in cve_train.columns
epss_tr = cve_train["epss_score"].values.reshape(-1, 1) if has_epss else np.zeros((len(cve_train), 1))
epss_te = cve_test["epss_score"].values.reshape(-1, 1) if has_epss else np.zeros((len(cve_test), 1))
X_train_hybrid = np.hstack([cve_train_emb, epss_tr])
X_test_hybrid = np.hstack([cve_test_emb, epss_te])
print(f"Hybrid features: {X_train_hybrid.shape[1]} dims")

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

cve_hybrid = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear"),
    method="sigmoid", cv=5
)
cve_hybrid.fit(X_train_hybrid, y_train_bin)

# Find optimal threshold on train
train_h_prob = cve_hybrid.predict_proba(X_train_hybrid)[:, 1]
best_ht, best_hf = 0.5, 0.0
for t in [x / 100 for x in range(5, 95)]:
    preds = (train_h_prob >= t).astype(int)
    f = f1_score(y_train_bin, preds, zero_division=0)
    if f > best_hf:
        best_hf, best_ht = f, t

test_h_prob = cve_hybrid.predict_proba(X_test_hybrid)[:, 1]
test_h_preds = (test_h_prob >= best_ht).astype(int)
print(f"=== CVE: MiniLM + EPSS + LogReg ===")
print(f"F1 at threshold {best_ht:.2f}: {f1_score(y_test_bin, test_h_preds):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test_bin, test_h_prob):.4f}")

# Ensemble: TF-IDF LogReg + Hybrid, weights tuned on train
lr_train_probs = cve_lr.predict_proba(cve_train["cve_text"])[:, 1]
ens_f1, ens_w, ens_t = 0.0, 0.5, 0.5
for w in [x / 20 for x in range(0, 21)]:
    ens_tr = w * lr_train_probs + (1 - w) * train_h_prob
    for t in [x / 100 for x in range(5, 95)]:
        preds = (ens_tr >= t).astype(int)
        f = f1_score(y_train_bin, preds, zero_division=0)
        if f > ens_f1:
            ens_f1, ens_w, ens_t = f, w, t

ens_test = ens_w * test_probs + (1 - ens_w) * test_h_prob
ens_preds = (ens_test >= ens_t).astype(int)
print(f"=== CVE: Ensemble (TF-IDF + MiniLM/EPSS) ===")
print(f"Weights: TF-IDF={ens_w:.2f}, Hybrid={1-ens_w:.2f}, threshold={ens_t:.2f}")
print(f"F1: {f1_score(y_test_bin, ens_preds):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test_bin, ens_test):.4f}")

joblib.dump({
    "lr_model": cve_lr,
    "hybrid_model": cve_hybrid,
    "lr_weight": ens_w,
    "hybrid_weight": 1 - ens_w,
    "decision_threshold": ens_t,
}, ARTIFACT_DIR / "cve_tfidf_logreg.joblib")
print("Saved: cve_tfidf_logreg.joblib")


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

CVE MiniLM embedding shape: train=(1268, 384), test=(317, 384)
Hybrid features: 385 dims


=== CVE: MiniLM + EPSS + LogReg ===
F1 at threshold 0.30: 0.4496
AUC-ROC: 0.7752


=== CVE: Ensemble (TF-IDF + MiniLM/EPSS) ===
Weights: TF-IDF=0.95, Hybrid=0.05, threshold=0.36
F1: 0.4906
AUC-ROC: 0.7557


Saved: cve_tfidf_logreg.joblib


### Domains Modeling ###

In [17]:
DOMAIN_TARGET = "Threat_Severity"

# Feature columns expected from preprocessing
domain_num_cols = [
    "Domain_Length", "Reputation", "Malicious_Votes",
    "Suspicious_Votes", "Harmless_Votes", "Undetected_Votes",
    "Total_Engines", "domain_age_days", "log_domain_age",
    "malicious_ratio", "suspicious_ratio",
    "log_malicious", "log_suspicious",
    "entropy", "digit_ratio", "vowel_ratio",
    "special_ratio", "subdomain_count", "token_count",
    "max_token_length", "consecutive_consonants", "consecutive_digits",
    "suspicious_keyword_count", "tld_risk_score",
    "whois_field_count",
]
domain_cat_cols = [
    "has_creation_date", "has_registrar", "has_nameservers",
    "contains_brand_keyword", "contains_login_keyword",
    "contains_crypto_keyword", "contains_bank_keyword",
    "is_randomized_domain", "is_new_domain",
    "Has_Numbers", "Has_Hyphen", "TLD", "Categories",
]

# Binary classification: merge Medium+High -> SUSPICIOUS
# Keep CLEAN as CLEAN
def to_binary(x):
    x = str(x).strip().lower()
    return 1 if x in ("medium", "high") else 0  # 1=SUSPICIOUS, 0=CLEAN

y_train = dom_train[DOMAIN_TARGET].apply(to_binary)
y_test = dom_test[DOMAIN_TARGET].apply(to_binary)

print(f"Train class distribution: {y_train.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")


Train class distribution: {0: 115, 1: 14}
Test class distribution: {0: 30, 1: 3}


In [18]:
# Limit top TLD categories to reduce sparsity
def limit_top_categories(df, col, top_k = 10):
    top = df[col].value_counts().head(top_k).index
    df[col] = df[col].where(df[col].isin(top), "other")
    return df

dom_train = limit_top_categories(dom_train.copy(), "TLD", top_k=10)
dom_test = limit_top_categories(dom_test.copy(), "TLD", top_k=10)

# Domain preprocessor
available_num = [c for c in domain_num_cols if c in dom_train.columns]
available_cat = [c for c in domain_cat_cols if c in dom_train.columns]

domain_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), available_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
    ]), available_cat)
], remainder="drop")

# LGBM with SMOTE-like class weighting (scale_pos_weight handles imbalance)
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
scale_weight = neg_count / max(pos_count, 1)

domain_model = Pipeline([
    ("prep", domain_preprocessor),
    ("clf", CalibratedClassifierCV(
        estimator=lgb.LGBMClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=2.0, reg_alpha=0.1,
            min_child_weight=2, num_leaves=31,
            scale_pos_weight=scale_weight,
            random_state=42, verbose=-1
        ),
        method="sigmoid", cv=5
    ))
])

# Train
domain_model.fit(dom_train, y_train)

# Evaluate with lower threshold for SUSPICIOUS (recall > precision for security)
probs = domain_model.predict_proba(dom_test)
probs_suspicious = probs[:, 1] if probs.shape[1] == 2 else probs[:, 0]

# Search best threshold on training data
train_probs = domain_model.predict_proba(dom_train)
train_probs_susp = train_probs[:, 1] if train_probs.shape[1] == 2 else train_probs[:, 0]

best_f1, best_thresh = 0.0, 0.5
for t in [x / 100 for x in range(10, 95)]:
    p = (train_probs_susp >= t).astype(int)
    f = f1_score(y_train, p, zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, t

preds = (probs_suspicious >= best_thresh).astype(int)

print(f"=== Domains: LGBM (binary, threshold={best_thresh:.2f}) ===")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Macro F1: {f1_score(y_test, preds, average='macro', zero_division=0):.4f}")
print(f"SUSPICIOUS Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(classification_report(y_test, preds, target_names=['CLEAN', 'SUSPICIOUS'], zero_division=0))

# Save
joblib.dump({
    "model": domain_model,
    "threshold": best_thresh,
}, ARTIFACT_DIR / "domain_model.joblib")
print("Saved: domain_model.joblib")


=== Domains: LGBM (binary, threshold=0.22) ===
Accuracy: 0.9697
Macro F1: 0.9201
SUSPICIOUS Recall: 1.0000
              precision    recall  f1-score   support

       CLEAN       1.00      0.97      0.98        30
  SUSPICIOUS       0.75      1.00      0.86         3

    accuracy                           0.97        33
   macro avg       0.88      0.98      0.92        33
weighted avg       0.98      0.97      0.97        33

Saved: domain_model.joblib


### IP Modeling ###

In [19]:
# Features (No Threat_Label/Category — they leak the target)
ip_num_cols = [
    "Malicious_Votes", "Suspicious_Votes",
    "Harmless_Votes", "Undetected_Votes",
    "Reputation_Score",
    "malicious_ratio", "suspicious_ratio",
    "log_malicious", "log_suspicious",
    "log_harmless", "log_undetected",
    "reputation_score_scaled",
    "Last_Analysis_Date_year",
    "Last_Analysis_Date_month",
    "Last_Analysis_Date_day",
]

ip_cat_cols = [
    "Country", "Continent", "ASN", "Owner", "Network",
    "Regional_Registry",
    "tor_flag", "ip_first_octet",
    "high_risk_country", "unknown_continent",
    "negative_reputation", "zero_votes", "asn_risk_flag",
]

available_num = [c for c in ip_num_cols if c in ip_train.columns]
available_cat = [c for c in ip_cat_cols if c in ip_train.columns]

X_train = ip_train[available_num + available_cat].copy()
X_test = ip_test[available_num + available_cat].copy()

# Target
y_train = ip_train["Threat_Severity"].astype(str).str.lower()
y_test = ip_test["Threat_Severity"].astype(str).str.lower()

le_ip = LabelEncoder()
y_train_enc = le_ip.fit_transform(y_train)

y_test = y_test.apply(lambda x: x if x in le_ip.classes_ else le_ip.classes_[0])
y_test_enc = le_ip.transform(y_test)

In [20]:
# Preprocessor
ip_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), available_num),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
    ]), available_cat)
])

In [21]:
# Evaluation helper with per-threshold HIGH class recall
def evaluate(name, y_true, preds, probs=None):
    """Evaluate with per-class metrics and HIGH-class recall at multiple thresholds."""
    print(f"\n===== {name} =====")
    print(f"Accuracy: {accuracy_score(y_true, preds):.4f}")
    print(f"Macro F1: {f1_score(y_true, preds, average='macro', zero_division=0):.4f}")
    print(classification_report(y_true, preds, target_names=le_ip.classes_, zero_division=0))

    # Per-class recall for HIGH class at lower thresholds
    high_idx = list(le_ip.classes_).index("high") if "high" in le_ip.classes_ else -1
    if high_idx >= 0 and probs is not None:
        print("  HIGH class recall @ different thresholds:")
        for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
            adjusted = (probs[:, high_idx] >= t).astype(int)
            high_recall = recall_score((y_true == high_idx).astype(int), adjusted, zero_division=0)
            print(f"    threshold={t:.1f} -> recall={high_recall:.4f}")


In [22]:
# Model 1: Logistic Regression (Calibrated)
logreg_model = Pipeline([
    ("prep", ip_preprocessor),
    ("clf", CalibratedClassifierCV(
        estimator=LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs"
        ),
        method="sigmoid",
        cv=5
    ))
])

logreg_model.fit(X_train, y_train_enc)
logreg_preds = logreg_model.predict(X_test)

In [23]:
# Results
evaluate("Logistic Regression", y_test_enc, logreg_preds)


===== Logistic Regression =====
Accuracy: 0.9250
Macro F1: 0.6530
              precision    recall  f1-score   support

        high       1.00      1.00      1.00         2
         low       0.92      1.00      0.96        35
      medium       0.00      0.00      0.00         3

    accuracy                           0.93        40
   macro avg       0.64      0.67      0.65        40
weighted avg       0.86      0.93      0.89        40



In [24]:
# Model 2: XGBoost (Calibrated) with bootstrap confidence intervals
xgb_model = Pipeline([
    ("prep", ip_preprocessor),
    ("clf", CalibratedClassifierCV(
        estimator=XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_lambda=2.0, reg_alpha=0.1,
            min_child_weight=2, random_state=42, eval_metric="mlogloss"
        ),
        method="sigmoid", cv=5
    ))
])

xgb_model.fit(X_train, y_train_enc)
xgb_probs = xgb_model.predict_proba(X_test)
xgb_preds = xgb_model.predict(X_test)

evaluate("XGBoost", y_test_enc, xgb_preds, xgb_probs)

# Bootstrap confidence intervals (100 resamples)
n_bootstrap = 100
bootstrap_scores = []
rng = np.random.RandomState(42)
for _ in range(n_bootstrap):
    idx = rng.randint(0, len(X_test), len(X_test))
    if len(np.unique(y_test_enc[idx])) > 1:
        bs_preds = xgb_model.predict(X_test.iloc[idx])
        bootstrap_scores.append(f1_score(y_test_enc[idx], bs_preds, average="macro", zero_division=0))

if bootstrap_scores:
    ci_low = np.percentile(bootstrap_scores, 2.5)
    ci_high = np.percentile(bootstrap_scores, 97.5)
    print(f"\nBootstrap 95% CI for Macro F1: [{ci_low:.4f}, {ci_high:.4f}]")

# Save both models (LogReg from cell above + XGBoost)
joblib.dump(
    {"model": xgb_model, "label_encoder": le_ip,
     "num_cols": available_num, "cat_cols": available_cat},
    ARTIFACT_DIR / "ip_xgb_model.joblib"
)
print("Saved: ip_xgb_model.joblib")

joblib.dump(
    {"model": logreg_model, "label_encoder": le_ip,
     "num_cols": available_num, "cat_cols": available_cat},
    ARTIFACT_DIR / "ip_logreg_model.joblib"
)
print("Saved: ip_logreg_model.joblib")



===== XGBoost =====
Accuracy: 1.0000
Macro F1: 1.0000
              precision    recall  f1-score   support

        high       1.00      1.00      1.00         2
         low       1.00      1.00      1.00        35
      medium       1.00      1.00      1.00         3

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

  HIGH class recall @ different thresholds:
    threshold=0.1 -> recall=1.0000
    threshold=0.2 -> recall=1.0000
    threshold=0.3 -> recall=1.0000
    threshold=0.4 -> recall=1.0000
    threshold=0.5 -> recall=1.0000



Bootstrap 95% CI for Macro F1: [1.0000, 1.0000]
Saved: ip_xgb_model.joblib
Saved: ip_logreg_model.joblib
